# バイオ技術 2-1：機械学習の基礎

このNotebookでは、乳がん診断データを使って、**機械学習による分類**を体験します。

今回はたくさんのモデルを試すのではなく、次の2つに絞ります。

- Logistic Regression
- Decision Tree

特にDecision Treeでは、

- 木構造を可視化する
- どの特徴量で判定しているかを見る
- 1症例がどの枝を通って判定されたか追跡する
- 木を複雑にしたときに何が起こるか考える

ことを中心にします。

---

## 今日のゴール

このNotebookが終わるころには、次のことを説明できることを目指します。

1. 機械学習とは何か
2. 分類と回帰の違い
3. 説明変数`X`と目的変数`y`の意味
4. Training dataとTest dataを分ける理由
5. Logistic RegressionとDecision Treeの違い
6. Decision Treeの読み方
7. 過学習とは何か

---

## このNotebookの進め方

途中で、予想したり、数字を書き換えたり、結果を考えたりします。

**実行する → 結果を見る → 自分で考える**

を意識して進めてください。


---
## 0. AI、機械学習、Deep Learning

AIという言葉は広い意味で使われます。

大まかには、

- AI
  - Machine Learning
    - Deep Learning

という関係で考えることができます。

今回の2-1では、表形式データに対する機械学習を扱います。  
次の2-2では、Deep Learningの一種であるCNNを使って画像分類を行います。

### 機械学習とは？

人がすべての判断ルールを書く代わりに、**データから判定パターンを学習する方法**です。

今回の例では、

> 細胞核の特徴量から、乳がんが良性か悪性かを予測する

ことを目指します。


---
## 1. 分類と回帰

教師あり学習には、代表的に次の2種類があります。

### Classification（分類）

カテゴリーを予測します。

例：

- 良性 / 悪性
- 陽性 / 陰性
- A群 / B群 / C群

### Regression（回帰）

連続した数値を予測します。

例：

- 血糖値
- 薬物濃度
- 疾患進行度

今回の2-1では**分類**を扱います。  
2-3では**回帰**を扱います。


---
## 2. 乳がんデータセットを読み込む

scikit-learnに付属しているBreast Cancer Wisconsin datasetを使います。

このデータには、細胞核画像から計算された複数の特徴量と、良性・悪性のラベルが含まれています。

今回の研究上の問いは、

> **細胞核の特徴量から、良性か悪性かを予測できるか？**

です。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

print("Data shape:", df.shape)
display(df.head())


### データの大きさを確認する

- 行：症例
- 列：特徴量と目的変数

です。

下のセルで、症例数と特徴量数を確認します。


In [ ]:
n_samples = df.shape[0]
n_features = df.shape[1] - 1

print("Number of samples:", n_samples)
print("Number of features:", n_features)


### 目的変数`target`を確認する

このデータでは、

- `0` = malignant
- `1` = benign

です。


In [ ]:
target_table = (
    df["target"]
    .value_counts()
    .sort_index()
    .rename(index={0: "malignant", 1: "benign"})
)

display(target_table.to_frame("count"))


### 考えてみよう

良性と悪性のデータ数は完全に同じではありません。

質問：

- どちらのクラスが多いですか？
- もし99%が良性のデータで、すべて「良性」と答えるモデルを作ったらAccuracyは何%になりますか？
- Accuracyだけを見ることの問題点はありそうですか？

答え：

-
-
-


---
## 3. 説明変数Xと目的変数y

機械学習では、入力する情報を`X`、予測したいものを`y`と書くことがよくあります。

今回の場合、

- `X`：細胞核の特徴量
- `y`：良性か悪性か

です。


In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())


---
## 4. まず人間がデータを見てみる

機械学習に任せる前に、いくつかの特徴量を人間の目で見てみましょう。

ここでは、

- mean radius
- mean texture

を良性・悪性で比較します。


In [ ]:
plot_df = df[["mean radius", "mean texture", "target"]].copy()
plot_df["diagnosis"] = plot_df["target"].map({0: "malignant", 1: "benign"})

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=plot_df,
    x="mean radius",
    y="mean texture",
    hue="diagnosis",
    alpha=0.7
)
plt.title("Breast cancer data")
plt.show()


### 考えてみよう

散布図を見てください。

1. 良性と悪性は完全に分離していますか？
2. `mean radius`が大きいほど、どちらのクラスが多そうですか？
3. 2つの特徴量だけで、すべての症例を完全に分類できそうですか？

答え：

-
-
-


---
## 5. Training dataとTest dataに分ける

機械学習では、同じデータで学習と評価をしてはいけません。

そこでデータを、

- Training data：モデルを学習するためのデータ
- Test data：未知データに対する性能を確認するためのデータ

に分けます。

今回は70%をTraining data、30%をTest dataにします。


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


### 確認問題

なぜTest dataを学習に使ってはいけないのでしょうか？

答え：


---
# Part 1：Logistic Regression

まず、シンプルな基準モデルとしてLogistic Regressionを使います。

Logistic Regressionは名前にRegressionとありますが、2クラス分類にもよく使われます。

ここでは、

> 複数の特徴量を組み合わせて、良性・悪性を予測する

という流れを体験します。


## 6. Logistic Regressionを学習する

Logistic Regressionでは特徴量の尺度の違いが影響するため、StandardScalerで標準化してから学習します。

Pipelineを使って、

1. 標準化
2. Logistic Regression

を順番に実行します。


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000, random_state=42)
)

lr_model.fit(X_train, y_train)

print("Logistic Regression training complete.")


## 7. Test dataを予測する


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

lr_pred = lr_model.predict(X_test)

lr_train_acc = accuracy_score(y_train, lr_model.predict(X_train))
lr_test_acc = accuracy_score(y_test, lr_pred)

print(f"Train accuracy: {lr_train_acc:.3f}")
print(f"Test accuracy : {lr_test_acc:.3f}")


Accuracyだけでなく、Confusion Matrixも確認します。

行が実際のクラス、列が予測クラスです。


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    lr_pred,
    display_labels=data.target_names,
    cmap="Blues"
)

plt.title("Logistic Regression")
plt.show()


### 考えてみよう

Confusion Matrixを見てください。

- 悪性を良性と予測した症例はいくつありましたか？
- 良性を悪性と予測した症例はいくつありましたか？
- 医療分野では、2種類の誤りを同じように考えてよいでしょうか？

答え：

-
-
-


---
# Part 2：Decision Tree

ここからDecision Treeを使います。

Decision Treeは、

> ある特徴量が閾値以下か？

という質問を繰り返しながら、最終的な分類を決めます。

人間にも判断の流れが見えやすいことが特徴です。


## 8. 浅いDecision Treeを作る

最初は`max_depth=3`として、木を浅く制限します。

浅い木は性能が少し低くなることがありますが、全体を人間が読みやすくなります。


In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)

tree_train_acc = accuracy_score(y_train, tree_model.predict(X_train))
tree_test_acc = accuracy_score(y_test, tree_pred)

print(f"Train accuracy: {tree_train_acc:.3f}")
print(f"Test accuracy : {tree_test_acc:.3f}")


## 9. Decision Treeを可視化する

各ノードには、主に次の情報が表示されます。

- 条件：例 `worst radius <= 16.8`
- samples：そのノードに到達したデータ数
- value：各クラスのデータ数
- class：そのノードで予測されるクラス

### 木の読み方

条件を満たす場合は**左**へ、満たさない場合は**右**へ進みます。


In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(22, 10))

plot_tree(
    tree_model,
    feature_names=X.columns,
    class_names=data.target_names,
    filled=True,
    rounded=True,
    fontsize=9
)

plt.title("Decision Tree (max_depth = 3)")
plt.show()


### ミニ演習1：木を読んでみよう

上のDecision Treeを見て答えてください。

1. 最初に使われている特徴量は何ですか？
2. 最初の条件を満たした場合、左と右のどちらに進みますか？
3. 左端の葉では、どちらのクラスが多いですか？
4. この判断ルールは、人間が手で設定したものでしょうか？それともTraining dataから学習されたものでしょうか？

答え：

1.
2.
3.
4.


## 10. 1つの症例が通る経路を追跡する

Decision Treeの面白いところは、1症例がどの条件分岐を通ったか追跡できることです。

まず、Test dataから1症例を選びます。


In [ ]:
# ↓↓↓ 0〜len(X_test)-1 の範囲で数字を変更できます ↓↓↓
SAMPLE_INDEX = 0

sample = X_test.iloc[[SAMPLE_INDEX]]
true_label = int(y_test.iloc[SAMPLE_INDEX])
pred_label = int(tree_model.predict(sample)[0])

print("Selected test sample index:", SAMPLE_INDEX)
print("True label      :", data.target_names[true_label])
print("Predicted label :", data.target_names[pred_label])


次のセルでは、この症例が通った経路を文章で表示します。


In [ ]:
from sklearn.tree import _tree

def show_decision_path(model, sample_df, feature_names):
    tree_ = model.tree_
    node_indicator = model.decision_path(sample_df)
    leaf_id = model.apply(sample_df)

    node_index = node_indicator.indices[
        node_indicator.indptr[0]:node_indicator.indptr[1]
    ]

    print("Decision path")
    print("-" * 60)

    for node_id in node_index:
        if node_id == leaf_id[0]:
            predicted_class = np.argmax(tree_.value[node_id][0])
            print(
                f"Leaf node {node_id}: "
                f"predict {data.target_names[predicted_class]}"
            )
            continue

        feature_index = tree_.feature[node_id]
        threshold = tree_.threshold[node_id]
        feature_name = feature_names[feature_index]
        sample_value = sample_df.iloc[0, feature_index]

        if sample_value <= threshold:
            direction = "LEFT"
            relation = "<="
        else:
            direction = "RIGHT"
            relation = ">"

        print(
            f"Node {node_id}: {feature_name} = {sample_value:.3f} "
            f"{relation} {threshold:.3f}  →  {direction}"
        )

show_decision_path(
    tree_model,
    sample,
    X.columns.to_list()
)


### ミニ演習2：別の症例を追跡する

上の`SAMPLE_INDEX`を変更して、少なくとも次の2種類を探してください。

- 正しく予測された症例
- 誤って予測された症例

見つけたら記録してください。

- 正しく予測されたSAMPLE_INDEX：
- 誤分類されたSAMPLE_INDEX：
- 誤分類症例は、どのような条件分岐を通っていましたか？：


---
## 11. Confusion Matrixを確認する

Decision Treeの予測結果も、Confusion Matrixで確認します。


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    tree_pred,
    display_labels=data.target_names,
    cmap="Blues"
)

plt.title("Decision Tree")
plt.show()


---
# Part 3：木を複雑にするとどうなるか？

Decision Treeは、深くすれば複雑なルールを作れます。

では、木を深くすれば、Test dataの性能も必ず良くなるのでしょうか？

ここで**過学習（overfitting）**を考えます。


## 12. max_depthを変えて比較する

次の4つを比較します。

- 2
- 3
- 5
- None（制限なし）

まず予想してください。

> max_depthを大きくすると、Train accuracyはどうなると思いますか？

予想：

> Test accuracyも同じように上がり続けると思いますか？

予想：


In [ ]:
depth_settings = [2, 3, 5, None]
depth_results = []

for depth in depth_settings:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )
    model.fit(X_train, y_train)

    train_acc = accuracy_score(
        y_train,
        model.predict(X_train)
    )

    test_acc = accuracy_score(
        y_test,
        model.predict(X_test)
    )

    depth_results.append({
        "max_depth": str(depth),
        "train_accuracy": train_acc,
        "test_accuracy": test_acc
    })

depth_df = pd.DataFrame(depth_results)
display(depth_df)


結果をグラフでも確認します。


In [ ]:
plot_depth_df = depth_df.melt(
    id_vars="max_depth",
    value_vars=["train_accuracy", "test_accuracy"],
    var_name="dataset",
    value_name="accuracy"
)

plt.figure(figsize=(7, 5))
sns.barplot(
    data=plot_depth_df,
    x="max_depth",
    y="accuracy",
    hue="dataset"
)

plt.ylim(0.80, 1.02)
plt.title("Effect of tree depth")
plt.show()


### ミニ演習3：過学習を考える

表とグラフを見て答えてください。

1. max_depthが大きくなるとTrain accuracyはどうなりましたか？
2. Test accuracyも同じように上がり続けましたか？
3. Training dataには非常によく合うが、未知のTest dataでは性能が落ちる現象を何と呼びますか？
4. 複雑なモデルほど必ず良いモデルだと言えるでしょうか？

答え：

1.
2.
3.
4.


---
## 13. Logistic RegressionとDecision Treeを比較する

最後に、2つのモデルを比較します。

ここで重要なのは、Accuracyだけで「モデルの価値」がすべて決まるわけではないことです。

Decision Treeには、

- 判断ルールが見やすい
- 1症例の判定経路を追跡できる

という特徴があります。

Logistic Regressionには、

- シンプルな基準モデルとして使いやすい
- 小〜中規模の表形式データでも高い性能を示すことがある

という特徴があります。


In [ ]:
comparison_df = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "Decision Tree (depth=3)"
    ],
    "train_accuracy": [
        lr_train_acc,
        tree_train_acc
    ],
    "test_accuracy": [
        lr_test_acc,
        tree_test_acc
    ]
})

display(comparison_df)


### 最後の考察

次の問いに自分の言葉で答えてください。

**Q1. 今回のデータでは、どちらのモデルのTest accuracyが高かったですか？**

答え：

**Q2. Decision Treeを使う利点は何だと思いますか？**

答え：

**Q3. 深いDecision TreeでTrain accuracyが高くても、必ずしも良いモデルとは言えないのはなぜですか？**

答え：

**Q4. 医療や生命科学でAIモデルを使うとき、Accuracy以外に何を考える必要があると思いますか？**

答え：


---
## 14. まとめ

このNotebookでは、乳がんデータを使って機械学習の基本を学びました。

### 今日の流れ

1. データを見る
2. 説明変数`X`と目的変数`y`を分ける
3. Training dataとTest dataに分割する
4. Logistic Regressionを学習する
5. Decision Treeを学習する
6. Treeを可視化する
7. 1症例の判定経路を追跡する
8. max_depthを変えて過学習を考える

### 次のNotebookへ

ここでは、表形式データを使いました。

次の2-2では、

> **画像をAIはどのように学習するのか？**

をMNIST画像とConvolutional Neural Network（CNN）を使って体験します。
